In [1]:
import os
import json
import csv
from collections import Counter

# =====================================================
# CONFIGURAÇÕES
# =====================================================

INPUT_PATH = "../database/"

# Três pastas de saída
OUTPUT_PATH_TEST = "benchmark_prepared_test/"
OUTPUT_PATH_VAL = "benchmark_prepared_val/"
OUTPUT_PATH_TRAIN = "benchmark_prepared_train/"

# True  = manter apenas séries do comprimento MAIS FREQUENTE
#         (o tamanho que aparece em mais séries) e adicionar
#         "_filtered" ao name do dataset
# False = manter todas as séries
FILTER_MAX_LENGTH = True

# Número alvo de sub-séries ao quebrar CSVs por id
TARGET_NUM_SERIES = 100

# Comprimento mínimo absoluto de uma série no split train (após corte 2*H)
# Séries que ficarem abaixo disso são removidas de TODOS os splits
MIN_SERIES_LENGTH = 32

for path in [OUTPUT_PATH_TEST, OUTPUT_PATH_VAL, OUTPUT_PATH_TRAIN]:
    os.makedirs(path, exist_ok=True)

# =====================================================
# DATASETS TSF (formato original)
# =====================================================

DATASETS_TSF = {
    "m3_monthly_dataset.tsf": {
        "name": "m3_monthly",
        "horizon": 18
    },
    "m4_monthly_dataset.tsf": {
        "name": "m4_monthly",
        "horizon": 18
    },
    "nn5_weekly_dataset.tsf": {
        "name": "nn5_weekly",
        "horizon": 8
    },
    "tourism_monthly_dataset.tsf": {
        "name": "tourism_monthly",
        "horizon": 24
    },
    "cif_2016_dataset.tsf": {
        "name": "cif_2016",
        "horizon": 12
    },
    "hospital_dataset.tsf": {
        "name": "hospital",
        "horizon": 12
    }
}

# =====================================================
# DATASETS CSV (novos)
# =====================================================

DATASETS_CSV = {
    "ETTh.csv": {
        "name": "etth",
        "horizon": 36,
        "target_column": "OT",
        "id_column": "id",
        "date_column": "date",
        "num_series": TARGET_NUM_SERIES
    },
    "weather.csv": {
        "name": "weather",
        "horizon": 36,
        "target_column": "OT",
        "id_column": None,
        "date_column": "date",
        "num_series": TARGET_NUM_SERIES
    }
}


# =====================================================
# PARSE DO ARQUIVO TSF
# =====================================================

def parse_tsf(file_path):

    sequences = []
    horizons = []

    with open(file_path, "r", encoding="latin1") as f:

        data_started = False

        for line in f:

            line = line.strip()

            if not line:
                continue

            if line.startswith("@data"):
                data_started = True
                continue

            if not data_started:
                continue

            parts = line.split(":")

            if len(parts) == 4:
                _, _, horizon, values = parts
                horizon = int(horizon)

            elif len(parts) == 3:
                _, _, values = parts
                horizon = None

            else:
                continue

            series = []

            for v in values.split(","):

                v = v.strip()

                if v == "?" or v == "":
                    continue

                try:
                    series.append(float(v))
                except:
                    continue

            if len(series) > 0:
                sequences.append(series)
                horizons.append(horizon)

    return sequences, horizons


# =====================================================
# PARSE DO ARQUIVO CSV
# =====================================================

def parse_csv(file_path, target_column, id_column, date_column):

    groups = {}

    with open(file_path, "r", encoding="utf-8") as f:

        reader = csv.DictReader(f)

        for row in reader:

            if id_column and id_column in row:
                gid = row[id_column].strip()
            else:
                gid = "all"

            raw_value = row[target_column].strip()

            if raw_value == "" or raw_value == "?":
                continue

            try:
                value = float(raw_value)
            except:
                continue

            date_str = row[date_column].strip() if date_column else ""

            if gid not in groups:
                groups[gid] = []

            groups[gid].append((date_str, value))

    for gid in groups:
        groups[gid].sort(key=lambda x: x[0])

    return groups


# =====================================================
# QUEBRAR SÉRIE EM N SUB-SÉRIES
# =====================================================

def split_series(values, num_series):

    total = len(values)

    if total == 0:
        return []

    chunk_size = total // num_series

    if chunk_size < 2:
        chunk_size = 2

    actual_num = total // chunk_size

    sub_series = []
    start = 0

    for i in range(actual_num):

        end = start + chunk_size
        sub_series.append(values[start:end])
        start = end

    if start < total:
        sub_series.append(values[start:total])

    return sub_series


# =====================================================
# FILTRAR SERIES PELO COMPRIMENTO MAIS FREQUENTE
# =====================================================

def filter_most_frequent_length(sequences):
    """
    Mantém apenas as séries cujo comprimento é o MAIS FREQUENTE
    no conjunto (o tamanho que aparece em mais séries).

    Ex.: se 20 séries têm tamanho 50 e 19 séries têm tamanho 60,
    retorna apenas as 20 séries de tamanho 50.

    Em caso de empate na frequência, escolhe o maior comprimento
    entre os mais frequentes.
    """

    if len(sequences) == 0:
        return sequences, 0

    length_counts = Counter(len(s) for s in sequences)

    max_count = max(length_counts.values())

    # Todos os comprimentos que atingem a frequência máxima
    most_frequent_lengths = [
        length for length, count in length_counts.items()
        if count == max_count
    ]

    # Desempate: maior comprimento
    chosen_length = max(most_frequent_lengths)

    filtered = [s for s in sequences if len(s) == chosen_length]

    return filtered, chosen_length


# =====================================================
# OBTER COMPRIMENTO MAXIMO
# =====================================================

def get_max_length(sequences):

    if len(sequences) == 0:
        return 0

    return max(len(s) for s in sequences)


# =====================================================
# FILTRAR SÉRIES VIÁVEIS PARA TRAIN
# =====================================================

def filter_viable_sequences(sequences, horizon, min_length=MIN_SERIES_LENGTH):
    """
    Mantém apenas séries que, após o corte mais agressivo
    (train = 2*horizon), ainda tenham pelo menos min_length pontos.

    A regra garante que:
      - Nenhuma série no train fique com menos de 32 pontos
      - Se uma série não sobrevive ao train, é removida de val e test também

    Isso garante consistência: a mesma série existe em todos os splits.
    """

    cut_train = horizon * 2

    viable = []
    removed = 0

    for seq in sequences:

        remaining = len(seq) - cut_train

        if remaining >= min_length:
            viable.append(seq)
        else:
            removed += 1

    return viable, removed


# =====================================================
# SALVAR JSONL
# =====================================================

def save_jsonl(sequences, output_file):

    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    with open(output_file, "w", encoding="utf-8") as f:

        for seq in sequences:

            json_line = {"sequence": seq}

            f.write(json.dumps(json_line) + "\n")


# =====================================================
# SALVAR NAS 3 PASTAS (train primeiro, depois val e test)
# =====================================================

def save_all_splits(sequences, dataset_name, horizon):
    """
    Salva o dataset nas 3 pastas.

    Ordem de criação:
      1. train: remove últimos 2*horizon pontos
      2. val:   remove últimos horizon pontos
      3. test:  série completa

    Apenas séries que sobrevivem ao corte do train são usadas
    em todos os splits (já filtradas antes de chamar esta função).
    """

    splits = [
        ("train", OUTPUT_PATH_TRAIN, 2),
        ("val",   OUTPUT_PATH_VAL,   1),
        ("test",  OUTPUT_PATH_TEST,  0),
    ]

    for split_name, output_base, multiplier in splits:

        cut = horizon * multiplier

        if cut == 0:
            trimmed = list(sequences)
        else:
            trimmed = [seq[:-cut] for seq in sequences]

        if len(trimmed) == 0:
            print(f"  WARNING: {split_name} - no series")
            continue

        output_file = os.path.join(
            output_base,
            dataset_name,
            f"horizon_{horizon}",
            "dataset.jsonl"
        )

        save_jsonl(trimmed, output_file)

        max_len = get_max_length(trimmed)
        min_len = min(len(s) for s in trimmed)

        print(
            f"  {split_name:5s} | "
            f"series={len(trimmed)} | "
            f"min_len={min_len} | "
            f"max_len={max_len} | "
            f"cut={cut}"
        )


# =====================================================
# PROCESSAMENTO TSF (original)
# =====================================================

print("=" * 60)
print("PROCESSANDO DATASETS TSF")
print("=" * 60)

for file_name, config in DATASETS_TSF.items():

    print(f"\nProcessing: {file_name}")

    dataset_path = os.path.join(INPUT_PATH, file_name)

    sequences, horizons = parse_tsf(dataset_path)

    dataset_name = config["name"]
    horizon = config["horizon"]

    total_series_original = len(sequences)

    # =================================================
    # CIF 2016
    # =================================================

    if dataset_name == "cif_2016":

        filtered_sequences = []

        for seq, h in zip(sequences, horizons):

            if h is None or h == 12:
                filtered_sequences.append(seq)

        sequences = filtered_sequences

    total_series_after_horizon_filter = len(sequences)

    if len(sequences) == 0:
        print("WARNING: No series found after filtering")
        continue

    # =================================================
    # FILTRAR PELO COMPRIMENTO MAIS FREQUENTE (OPCIONAL)
    # =================================================

    max_len = get_max_length(sequences)

    if FILTER_MAX_LENGTH:

        sequences, chosen_length = filter_most_frequent_length(sequences)

        print(
            f"  filter_most_frequent_length | "
            f"chosen_length={chosen_length} | "
            f"series_kept={len(sequences)}"
        )

        # Adiciona "_filtered" ao nome do dataset
        dataset_name = dataset_name + "_filtered"

        # Atualiza max_len após o filtro
        max_len = get_max_length(sequences)

    # =================================================
    # FILTRAR SÉRIES VIÁVEIS (sobrevivem ao corte train)
    # =================================================

    sequences, removed = filter_viable_sequences(sequences, horizon)

    total_series_final = len(sequences)

    if total_series_final == 0:
        print("WARNING: No viable series (all too short for train cut)")
        continue

    print(
        f"  original={total_series_original} | "
        f"after_horizon_filter={total_series_after_horizon_filter} | "
        f"viable={total_series_final} | "
        f"removed_too_short={removed} | "
        f"max_len={max_len} | "
        f"horizon={horizon}"
    )

    # =================================================
    # SALVAR NAS 3 PASTAS
    # =================================================

    save_all_splits(sequences, dataset_name, horizon)


# =====================================================
# PROCESSAMENTO CSV (novos datasets)
# =====================================================

print("\n" + "=" * 60)
print("PROCESSANDO DATASETS CSV")
print("=" * 60)

for file_name, config in DATASETS_CSV.items():

    print(f"\nProcessing: {file_name}")

    dataset_path = os.path.join(INPUT_PATH, file_name)

    dataset_name = config["name"]
    horizon = config["horizon"]
    target_column = config["target_column"]
    id_column = config["id_column"]
    date_column = config["date_column"]
    num_series = config["num_series"]

    # Ler CSV e agrupar por id
    groups = parse_csv(dataset_path, target_column, id_column, date_column)

    all_sequences = []

    for gid, records in groups.items():

        values = [v for _, v in records]

        total_points = len(values)
        chunk_size = total_points // num_series if total_points >= num_series else total_points

        print(
            f"  id={gid} | "
            f"total_points={total_points} | "
            f"target_num_series={num_series} | "
            f"chunk_size={chunk_size}"
        )

        sub_series = split_series(values, num_series)
        all_sequences.extend(sub_series)

    # =================================================
    # FILTRAR PELO COMPRIMENTO MAIS FREQUENTE (OPCIONAL)
    # =================================================

    if FILTER_MAX_LENGTH:

        all_sequences, chosen_length = filter_most_frequent_length(all_sequences)

        print(
            f"  filter_most_frequent_length | "
            f"chosen_length={chosen_length} | "
            f"series_kept={len(all_sequences)}"
        )

        # Adiciona "_filtered" ao nome do dataset
        dataset_name = dataset_name + "_filtered"

    # =================================================
    # FILTRAR SÉRIES VIÁVEIS (sobrevivem ao corte train)
    # =================================================

    all_sequences, removed = filter_viable_sequences(all_sequences, horizon)

    total_series_final = len(all_sequences)

    if total_series_final == 0:
        print("WARNING: No viable series generated")
        continue

    max_len = get_max_length(all_sequences)
    min_len = min(len(s) for s in all_sequences)

    print(
        f"  total_ids={len(groups)} | "
        f"viable_series={total_series_final} | "
        f"removed_too_short={removed} | "
        f"min_len={min_len} | "
        f"max_len={max_len} | "
        f"horizon={horizon}"
    )

    # =================================================
    # SALVAR NAS 3 PASTAS
    # =================================================

    save_all_splits(all_sequences, dataset_name, horizon)


print("\nAll datasets processed successfully.")

PROCESSANDO DATASETS TSF

Processing: m3_monthly_dataset.tsf
  filter_most_frequent_length | chosen_length=134 | series_kept=348
  original=1428 | after_horizon_filter=1428 | viable=348 | removed_too_short=0 | max_len=134 | horizon=18
  train | series=348 | min_len=98 | max_len=98 | cut=36
  val   | series=348 | min_len=116 | max_len=116 | cut=18
  test  | series=348 | min_len=134 | max_len=134 | cut=0

Processing: m4_monthly_dataset.tsf
  filter_most_frequent_length | chosen_length=87 | series_kept=6703
  original=48000 | after_horizon_filter=48000 | viable=6703 | removed_too_short=0 | max_len=87 | horizon=18
  train | series=6703 | min_len=51 | max_len=51 | cut=36
  val   | series=6703 | min_len=69 | max_len=69 | cut=18
  test  | series=6703 | min_len=87 | max_len=87 | cut=0

Processing: nn5_weekly_dataset.tsf
  filter_most_frequent_length | chosen_length=113 | series_kept=111
  original=111 | after_horizon_filter=111 | viable=111 | removed_too_short=0 | max_len=113 | horizon=8
  tra